# YOLO11s + 공통 분류기 · Kaggle 제출 CSV 만들기

**이 노트북은 CSV 생성까지만 합니다. Kaggle 업로드는 웹에서 직접 합니다.**

이미지만 있는 test 폴더가 정상입니다. 정답 JSON은 필요하지 않습니다. 모델이 test 사진을 보고 알약 위치와 약품 종류를 예측하고, Kaggle이 숨겨진 정답으로 채점합니다.

흐름: test 이미지 → YOLO → 알약 crop → 공통 ResNet50 → 팀 코드로 submission.csv 저장 → 다운로드 → Kaggle 업로드.

사용 모델은 val e2e 평가 때와 동일합니다. 재학습은 하지 않습니다. 새로운 Colab GPU 런타임에서 순서대로 실행하세요.


In [ ]:
import os, sys, subprocess
os.environ["WANDB_MODE"] = "disabled"
os.environ["YOLO_CONFIG_DIR"] = "/content/yolo-e2e-settings"
os.environ["NO_ALBUMENTATIONS_UPDATE"] = "1"
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
    "ultralytics==8.4.183", "torchmetrics[detection]==1.9.0", "pycocotools",
    "pandas", "matplotlib", "pyyaml", "pillow"])
import torch
assert torch.cuda.is_available(), "런타임 유형을 GPU로 바꾸세요."
print(torch.cuda.get_device_name(0), torch.__version__)


## 저장소 코드 준비·Drive 연결

저장소에서 직접 실행합니다. Colab에 체크아웃이 없으면 GitHub에서 가져옵니다. PR 병합 전 검토 시 SOURCE_REF를 작업 브랜치로 지정하세요.


In [ ]:
import sys, subprocess
from pathlib import Path
SOURCE_REF = "main"  # PR 검토 중에는 "37-yolo11s-training-e2e"로 지정
REPOSITORY_URL = "https://github.com/auranova1125/gitkk-pill-detection.git"
project_root = next((p for p in (Path.cwd(), *Path.cwd().parents)
                     if (p / "pyproject.toml").is_file()), None)
if project_root is None:
    project_root = Path("/content/gitkk-pill-detection")
    if not (project_root / ".git").is_dir():
        subprocess.check_call(["git", "clone", "--branch", SOURCE_REF, REPOSITORY_URL, str(project_root)])
PROJECT_ROOT = project_root.resolve()
assert (PROJECT_ROOT / "pipeline/yolo_e2e.py").is_file(), "YOLO 코드가 있는 브랜치를 열어주세요. PR 검토 중에는 SOURCE_REF를 변경하세요."
import os
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))
# 이전 프로젝트 복사본이 같은 Colab 세션에서 import된 경우 새 체크아웃 기준으로 읽습니다.
for name, module in list(sys.modules.items()):
    filename = str(getattr(module, "__file__", ""))
    if name.split(".")[0] in {"pipeline", "classifier", "detector", "data_loading", "configs", "utils"}:
        if "/content/" in filename and not filename.startswith(str(PROJECT_ROOT)):
            del sys.modules[name]
from google.colab import drive
drive.mount("/content/drive")


## 2. 입력 경로 설정
Test는 `raw_data/1. Kaggle/test_images`에서 읽습니다. 이미지 파일명은 기존 팀 추론 규칙대로 숫자 image ID여야 하며 코드가 검사합니다. 예: `123.png` → image_id=123. 이름이 다르면 임의로 번호를 바꾸지 말고 원본의 ID 규칙을 확인해야 합니다.

이번에는 이전 e2e의 주 표기 조건과 같이 **탐지 score ≥ 0.25**를 사용하고, 최종 제출 score는 탐지 score와 분류 확률의 곱으로 저장합니다. 팀 threshold 위치가 확정되면 맞춰 비교하세요.

같은 설정으로 재개할 때는 반드시 같은 RUN_NAME을 사용합니다. 설정이나 가중치를 바꾸는 실험은 실행 이름의 순번을 올립니다.


In [ ]:
from datetime import datetime
from zoneinfo import ZoneInfo
ARCHIVE = Path("/content/drive/MyDrive/gitkk_archive")
DETECTOR_RUN = "20261103-yolo11s-det-v2-baseline-01"
RUN_NAME = datetime.now(ZoneInfo("Asia/Seoul")).strftime("%Y%m%d") + "-yolo11s-resnet50-infer-test-submission-01"
# 재개할 때 위 줄을 기존 실행명 문자열로 바꿉니다. 예: RUN_NAME = "20261104-yolo11s-resnet50-infer-test-submission-01"
config = {
    "run_name": RUN_NAME,
    "detector_run": DETECTOR_RUN,
    "detector_checkpoint": str(ARCHIVE / "outputs/detector" / DETECTOR_RUN / "weights/best.pt"),
    "classifier_checkpoint": str(ARCHIVE / "models/classifier/resnet50_baseline_v2/best_v2_slim.pt"),
    "class_map": str(ARCHIVE / "models/classifier/resnet50_baseline_v2/class_map.json"),
    "input_dir": str(ARCHIVE / "raw_data/1. Kaggle/test_images"),
    "output_dir": str(ARCHIVE / "outputs/inference"),
    "image_cache_dir": "/content/kaggle-test-images",
    "device": "cuda:0", "detector_batch": 8, "classifier_batch": 64,
    "threshold_on": "detector", "sample_submission": None,
}
config.update({
    "expected_classifier_sha256": "d32d1f3030fe14ba5edf6b18e048f81cc2377e735ba61d22d3c87a7e97d90d69",
    "expected_class_map_sha256": "166ed118a34efce825a7999f4e85f2aa8746be6ebfeedd7afca08651f7fb2407",
    "expected_detector_sha256": "754e8e46b61a3aa372cc78e7cca2627fafbc44c6b377a74f08f73e681623ce92",
})
for key in ("detector_checkpoint", "classifier_checkpoint", "class_map"):
    path = Path(config[key])
    assert path.is_file(), f"파일을 찾을 수 없습니다: {path}"
    print(key, path.name, f"{path.stat().st_size / 1024**2:.2f} MiB")
assert Path(config["input_dir"]).is_dir(), f"test 경로를 확인하세요: {config['input_dir']}"
Path("outputs/runtime").mkdir(parents=True, exist_ok=True)
Path("outputs/runtime/kaggle-config.json").write_text(json.dumps(config, indent=2))
print("실행명:", RUN_NAME)
print("최종 결과 폴더:", Path(config["output_dir"]) / RUN_NAME)


## 3. 선택: sample_submission 컬럼 확인
Kaggle Data에서 sample_submission.csv를 받았다면 이 셀에서 업로드하세요. 없으면 건너뛸 수 있습니다. 코드가 같은 컬럼인지 대조합니다.

대회 페이지는 이 준비 과정에서 직접 열리지 않아 외부 제출 형식까지 확인하지 못했습니다. 기본 출력 형식은 팀이 이미 구현한 8개 컬럼 `annotation_id,image_id,category_id,bbox_x,bbox_y,bbox_w,bbox_h,score`입니다. Sample 파일이 있다면 대조하는 것이 좋습니다. 객체 수가 이미지마다 달라 sample과 행 개수를 같게 강제하지는 않습니다.


In [ ]:
# sample_submission.csv가 있으면 True로 바꿔 실행하세요. 없으면 False 그대로 둡니다.
UPLOAD_SAMPLE = False
if UPLOAD_SAMPLE:
    from google.colab import files
    uploaded = files.upload()
    sample_name = next((name for name in uploaded if name.endswith(".csv")), None)
    assert sample_name, "sample_submission.csv를 선택하세요."
    sample_path = Path("/content/sample_submission_reference.csv")
    sample_path.write_bytes(uploaded[sample_name])
    config["sample_submission"] = str(sample_path)
print("CSV 형식 대조용 sample:", config["sample_submission"])


## 4. Test 전체 추론 → submission.csv 생성
정답 파일을 읽거나 val 점수를 계산하지 않습니다. 기존 팀 함수 `save_submission_csv()`로 결과를 저장합니다. Val의 예측 CSV는 사용하지 않습니다.

Test 이미지를 로컬에 복사하므로 처음에는 시간이 걸립니다. 이미지별 추론 기록을 Drive에 저장해 같은 실행명으로 재개할 수 있습니다. 예측이 0개인 이미지에는 임의 박스를 추가하지 않고 개수를 따로 기록합니다.


In [ ]:
from pipeline.yolo_submission import run_test_submission
run_dir = run_test_submission(config)
print("제출 파일 생성 완료:", run_dir / "submission.csv")
print((run_dir / "submission_validation.json").read_text())


## 5. CSV 확인·다운로드 (런타임 초기화 후에도 실행 가능)
모델이나 test 이미지를 다시 읽지 않고 Drive의 완성된 CSV만 확인합니다. RUN_NAME_TO_VIEW를 비우면 해당 패턴의 검증 완료 실행 중 이름이 가장 뒤인 폴더를 선택합니다. 출력되는 실행 이름을 확인하세요.


In [ ]:
from pathlib import Path
from google.colab import drive, files
from IPython.display import display
import json, pandas as pd

drive.mount("/content/drive")
OUTPUT_ROOT = Path("/content/drive/MyDrive/gitkk_archive/outputs/inference")
RUN_NAME_TO_VIEW = ""  # 예: "20261104-yolo11s-resnet50-infer-test-submission-01"
if RUN_NAME_TO_VIEW:
    run_dir = OUTPUT_ROOT / RUN_NAME_TO_VIEW
else:
    completed = [p.parent for p in OUTPUT_ROOT.glob("*-yolo11s-resnet50-infer-test-submission-*/submission_validation.json")]
    assert completed, "완성된 test 제출 실행이 없습니다. 위 로그를 확인하세요."
    run_dir = sorted(completed, key=lambda p: p.name)[-1]
print("확인하는 실행:", run_dir.name)
summary = json.loads((run_dir / "submission_validation.json").read_text())
print(json.dumps(summary, ensure_ascii=False, indent=2))
submission = pd.read_csv(run_dir / "submission.csv")
display(submission.head(15))
assert list(submission.columns) == ["annotation_id", "image_id", "category_id", "bbox_x", "bbox_y", "bbox_w", "bbox_h", "score"]
assert submission["annotation_id"].is_unique
files.download(str(run_dir / "submission.csv"))


## 6. Kaggle 웹에서 제출

1. [대회 페이지](https://www.kaggle.com/competitions/ai15-level1-project/overview)에 로그인합니다.
2. 참가·규칙 동의가 필요하다면 먼저 진행합니다.
3. **Submit Predictions** 또는 제출 메뉴에서 방금 다운로드한 `submission.csv`를 선택합니다.
4. 메시지에 `YOLO11s best epoch20 + shared resnet50 v2; detector threshold 0.25`처럼 조건을 적습니다.
5. 제출 후 상태가 완료될 때까지 기다리고 표시되는 점수를 확인합니다. **Public score**라면 그렇게 기록합니다.

아래는 사용자가 알려준 CLI 명령의 형태가 맞지만, API 인증 설정이 필요하므로 이 노트북에서는 실행하지 않습니다. 웹 업로드만 해도 됩니다.

```bash
kaggle competitions submit -c ai15-level1-project -f submission.csv -m "YOLO11s + shared ResNet50 v2"
```

이 노트북은 Kaggle 계정 인증이나 자동 제출을 하지 않습니다. “CSV 생성 완료”와 “Kaggle 제출 완료”는 별도 단계입니다. 제출 오류가 뜨면 오류 문구를 기록하고 확인한 뒤 진행하세요.


## 7. Kaggle 점수를 받은 뒤 노션 통합 기록 만들기
아래 PUBLIC_SCORE에 화면에 표시된 값을 직접 넣으세요. 아직 점수가 안 나왔으면 실행하지 않습니다. Val은 reference COCO 평가이고 Kaggle Public은 숨겨진 test의 공개 점수이므로 구분해 기록합니다.


In [ ]:
from pathlib import Path
from google.colab import files
from IPython.display import display, Markdown
from datetime import datetime
from zoneinfo import ZoneInfo
import json, math

PUBLIC_SCORE = ""  # Kaggle에 실제로 표시된 값만 입력합니다.
SUBMISSION_TIME = ""  # Kaggle에 표시된 제출 시각
SUBMISSION_MESSAGE = "YOLO11s best epoch20 + shared resnet50 v2; detector threshold 0.25"
assert PUBLIC_SCORE.strip(), "Kaggle 채점이 끝난 뒤 실제 점수를 입력하세요."
assert math.isfinite(float(PUBLIC_SCORE))
record = {"public_score": float(PUBLIC_SCORE), "submission_time": SUBMISSION_TIME,
          "message": SUBMISSION_MESSAGE, "competition": "ai15-level1-project",
          "recorded_at_kst": datetime.now(ZoneInfo("Asia/Seoul")).isoformat(),
          "submission_file": str(run_dir / "submission.csv")}
(run_dir / "kaggle_result.json").write_text(json.dumps(record, ensure_ascii=False, indent=2))
val_path = Path("/content/drive/MyDrive/gitkk_archive/outputs/inference/20261104-yolo11s-resnet50-det-v2-evaluation-01/scores.json")
text = "# YOLO11s + 공통 ResNet50 · val 평가 및 Kaggle 제출\n\n"
text += "알약 위치를 찾는 YOLO와 약품 종류를 구별하는 공통 ResNet50을 연결했다. 두 모델을 다시 학습하지 않았다.\n\n"
text += "| 평가 | 결과 | 의미 |\n| --- | ---: | --- |\n"
if val_path.is_file():
    val = json.loads(val_path.read_text())
    text += f"| 전체 val e2e mAP@[0.75:0.95] | {val['all_val_detector']['map_75_95']:.6f} | 2,026장·참고용 COCO 평가 |\n"
    text += f"| 전체 val AP@0.95 | {val['all_val_detector']['ap_95']:.6f} | 엄격한 위치 기준 |\n"
text += f"| Kaggle Public score | {record['public_score']} | 숨겨진 test의 공개 리더보드 점수 |\n\n"
text += f"- 제출 실행: `{run_dir.name}`\n- 탐지기: `20261103-yolo11s-det-v2-baseline-01/weights/best.pt`\n- 공통 분류기: `best_v2_slim.pt`\n- 제출 시각: {SUBMISSION_TIME}\n- 제출 메시지: {SUBMISSION_MESSAGE}\n\n"
text += "Val과 Kaggle Public은 평가 데이터와 채점 구현이 다를 수 있어 같은 시험 점수처럼 단순 비교하지 않는다. Public 점수는 최종 Private 점수와 구분한다. 이번 기록은 화면에서 확인한 점수를 직접 입력한 것이다.\n\n"
text += "## 직접 확인하며 추가할 내용\n\n- 내가 예상한 점수와 달랐는가?:\n- 팀의 다른 탐지기와 같은 조건으로 비교했는가?:\n- 다음에 확인할 점:\n"
report = run_dir / "notion_val_and_kaggle.md"
report.write_text(text)
display(Markdown(text))
files.download(str(report))
